# MSR Project: Comprehensive Visualization Recreation
## AI Agents in Software Development - Visual Analytics Dashboard

This notebook recreates and enhances all visualizations from the MSR analysis pipeline, providing comprehensive visual insights into AI agent behavior in software development.

### 📊 Visualization Sections
1. **Agent Distribution Analysis** - Usage patterns and adoption metrics
2. **Test Contribution Behavior** - Testing practices across agents
3. **Test-to-Code Ratios** - Quality metrics and comparative analysis
4. **Code Change Characteristics** - Development pattern analysis
5. **Description Consistency** - Communication quality metrics
6. **User Adoption Trends** - Adoption patterns over time
7. **Executive Dashboard** - Key performance indicators
8. **Interactive Explorations** - Dynamic analysis tools

### 🎯 Key Questions Addressed
- Which AI agents are most widely adopted?
- How do testing practices vary between agents?
- What are the quality characteristics of AI-generated code?
- How consistent are AI-generated descriptions?

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import json
import os
import sys
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')
import importlib
if 'data_loader' in sys.modules:
    importlib.reload(sys.modules['data_loader'])
if 'analysis' in sys.modules:
    importlib.reload(sys.modules['analysis'])
from data_loader import load_aidev
from analysis import *

# Configure plotting styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Set up output directories
outputs_dir = Path('../outputs')
figures_dir = outputs_dir / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

print("🎨 MSR Visualization Recreation Toolkit")
print("=" * 50)
print(f"📁 Output directory: {figures_dir}")
print(f"🕒 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Ready to recreate comprehensive visualizations!")

In [ ]:
# Load and Prepare MSR Dataset
print("📊 Loading MSR dataset...")
df = load_aidev(sample_size=50000)  # Load comprehensive dataset
print(f"✅ Loaded {len(df):,} PRs for visualization")

# Apply analysis functions to prepare data
df, test_by_agent = analyze_test_contributions(df)

# Define visualization helper function
def calculate_test_code_ratios(df):
    """Calculate test-to-code ratios for visualization"""
    if 'is_test_pr' not in df.columns:
        df, _ = analyze_test_contributions(df)
    
    test_prs = df['is_test_pr'].sum()
    code_prs = len(df) - test_prs
    total_prs = len(df)
    test_ratio = test_prs / total_prs if total_prs > 0 else 0
    test_to_code_ratio = test_prs / code_prs if code_prs > 0 else float('inf')
    
    overall_stats = {
        'test_prs': test_prs,
        'code_prs': code_prs,
        'total_prs': total_prs,
        'test_ratio': test_ratio,
        'test_to_code_ratio': test_to_code_ratio
    }
    
    by_agent_stats = {}
    for agent in df['agent'].unique():
        agent_df = df[df['agent'] == agent]
        agent_test_prs = agent_df['is_test_pr'].sum()
        agent_code_prs = len(agent_df) - agent_test_prs
        agent_total_prs = len(agent_df)
        agent_test_ratio = agent_test_prs / agent_total_prs if agent_total_prs > 0 else 0
        agent_test_to_code_ratio = agent_test_prs / agent_code_prs if agent_code_prs > 0 else float('inf')
        
        by_agent_stats[agent] = {
            'test_prs': agent_test_prs,
            'code_prs': agent_code_prs,
            'total_prs': agent_total_prs,
            'test_ratio': agent_test_ratio,
            'test_to_code_ratio': agent_test_to_code_ratio
        }
    
    return {'overall': overall_stats, 'by_agent': by_agent_stats}

# Generate ratio data
ratio_data = calculate_test_code_ratios(df)

# Display dataset overview
print(f"\n📈 Dataset Overview:")
print(f"  • Total PRs: {len(df):,}")
print(f"  • Unique Agents: {df['agent'].nunique()}")
print(f"  • Date Range: {df['created_at'].min()} to {df['created_at'].max()}")
print(f"  • Test PR Rate: {df['is_test_pr'].mean():.1%}")

agent_counts = df['agent'].value_counts()
print(f"\n🤖 Top Agents by PR Count:")
for agent, count in agent_counts.head().items():
    percentage = (count / len(df)) * 100
    print(f"  • {agent}: {count:,} PRs ({percentage:.1f}%)")

## 1. Agent Distribution Analysis
### Comprehensive view of AI agent usage patterns and adoption metrics

In [ ]:
# Agent Distribution Visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 15))
fig.suptitle('🤖 AI Agent Distribution Analysis', fontsize=20, fontweight='bold')

# 1. Agent Usage Bar Chart
agent_counts = df['agent'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(agent_counts)))
bars = ax1.bar(range(len(agent_counts)), agent_counts.values, color=colors)
ax1.set_title('PR Count by AI Agent', fontsize=14, fontweight='bold')
ax1.set_xlabel('AI Agents')
ax1.set_ylabel('Number of PRs')
ax1.set_xticks(range(len(agent_counts)))
ax1.set_xticklabels(agent_counts.index, rotation=45, ha='right')

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, agent_counts.values)):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + len(df)*0.01,
             f'{count:,}', ha='center', va='bottom', fontweight='bold')

# 2. Agent Market Share Pie Chart
ax2.pie(agent_counts.values, labels=agent_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
ax2.set_title('Market Share by Agent', fontsize=14, fontweight='bold')

# 3. Agent Adoption Timeline (if date available)
if 'created_at' in df.columns:
    df['date'] = pd.to_datetime(df['created_at'])
    df['month'] = df['date'].dt.to_period('M')
    
    # Monthly trends by agent
    monthly_trends = df.groupby(['month', 'agent']).size().unstack(fill_value=0)
    
    for agent in monthly_trends.columns[:5]:  # Top 5 agents
        ax3.plot(monthly_trends.index.astype(str), monthly_trends[agent], 
                marker='o', linewidth=2, label=agent)
    
    ax3.set_title('Agent Adoption Trends Over Time', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Month')
    ax3.set_ylabel('Number of PRs')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.tick_params(axis='x', rotation=45)

# 4. Agent Efficiency Metrics (PRs per day)
if 'created_at' in df.columns:
    df['date_only'] = df['date'].dt.date
    daily_activity = df.groupby(['agent', 'date_only']).size().reset_index(name='prs_per_day')
    avg_daily_prs = daily_activity.groupby('agent')['prs_per_day'].mean().sort_values(ascending=False)
    
    bars = ax4.bar(range(len(avg_daily_prs)), avg_daily_prs.values, 
                   color=plt.cm.viridis(np.linspace(0, 1, len(avg_daily_prs))))
    ax4.set_title('Average Daily PR Activity by Agent', fontsize=14, fontweight='bold')
    ax4.set_xlabel('AI Agents')
    ax4.set_ylabel('Average PRs per Day')
    ax4.set_xticks(range(len(avg_daily_prs)))
    ax4.set_xticklabels(avg_daily_prs.index, rotation=45, ha='right')
    
    # Add value labels
    for bar, value in zip(bars, avg_daily_prs.values):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                 f'{value:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'agent_distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"📊 Agent Distribution Analysis Complete!")
print(f"📁 Saved: {figures_dir / 'agent_distribution_analysis.png'}")

## 2. Test Contribution Behavior Analysis
### Examining testing practices and quality assurance across AI agents

In [ ]:
# Test Contribution Analysis
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 15))
fig.suptitle('🧪 Test Contribution Behavior Analysis', fontsize=20, fontweight='bold')

# 1. Overall Test vs Non-Test Distribution
test_counts = df['is_test_pr'].value_counts()
labels = ['Non-Test PRs', 'Test PRs']
colors = ['lightcoral', 'lightgreen']
explode = (0.05, 0)  # Explode the test slice slightly

ax1.pie(test_counts.values, labels=labels, autopct='%1.1f%%', 
        colors=colors, explode=explode, startangle=90, shadow=True)
ax1.set_title('Overall Test Contribution Rate', fontsize=14, fontweight='bold')

# 2. Test Contribution Rate by Agent
test_rates = df.groupby('agent')['is_test_pr'].agg(['count', 'sum', 'mean']).reset_index()
test_rates['test_percentage'] = test_rates['mean'] * 100
test_rates = test_rates.sort_values('test_percentage', ascending=False)

bars = ax2.bar(range(len(test_rates)), test_rates['test_percentage'], 
               color=plt.cm.RdYlGn(test_rates['test_percentage']/100))
ax2.set_title('Test Contribution Rate by Agent (%)', fontsize=14, fontweight='bold')
ax2.set_xlabel('AI Agents')
ax2.set_ylabel('Test PR Percentage')
ax2.set_xticks(range(len(test_rates)))
ax2.set_xticklabels(test_rates['agent'], rotation=45, ha='right')

# Add value labels and overall average line
overall_test_rate = df['is_test_pr'].mean() * 100
ax2.axhline(y=overall_test_rate, color='red', linestyle='--', 
           label=f'Overall Average ({overall_test_rate:.1f}%)')
ax2.legend()

for bar, percentage in zip(bars, test_rates['test_percentage']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{percentage:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. Test vs Non-Test Volume Comparison
agent_test_data = df.groupby('agent')['is_test_pr'].agg(['sum', 'count']).reset_index()
agent_test_data['non_test'] = agent_test_data['count'] - agent_test_data['sum']
agent_test_data = agent_test_data.sort_values('count', ascending=False).head(8)

x = np.arange(len(agent_test_data))
width = 0.35

bars1 = ax3.bar(x - width/2, agent_test_data['sum'], width, 
                label='Test PRs', color='lightgreen', alpha=0.8)
bars2 = ax3.bar(x + width/2, agent_test_data['non_test'], width,
                label='Non-Test PRs', color='lightcoral', alpha=0.8)

ax3.set_title('Test vs Non-Test PR Volume by Agent', fontsize=14, fontweight='bold')
ax3.set_xlabel('AI Agents')
ax3.set_ylabel('Number of PRs')
ax3.set_xticks(x)
ax3.set_xticklabels(agent_test_data['agent'], rotation=45, ha='right')
ax3.legend()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 50,
                 f'{int(height):,}', ha='center', va='bottom', fontsize=10)

# 4. Test Quality Heatmap (Test Rate by Agent and Month)
if 'created_at' in df.columns:
    df['month'] = pd.to_datetime(df['created_at']).dt.to_period('M')
    test_heatmap = df.groupby(['agent', 'month'])['is_test_pr'].mean().unstack(fill_value=0)
    
    # Select top agents and recent months for clarity
    top_agents = df['agent'].value_counts().head(6).index
    recent_months = test_heatmap.columns[-12:]  # Last 12 months
    heatmap_data = test_heatmap.loc[top_agents, recent_months]
    
    im = ax4.imshow(heatmap_data.values, cmap='RdYlGn', aspect='auto')
    ax4.set_title('Test Rate Heatmap (Agent × Month)', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Month')
    ax4.set_ylabel('AI Agents')
    ax4.set_xticks(range(len(recent_months)))
    ax4.set_xticklabels([str(m) for m in recent_months], rotation=45, ha='right')
    ax4.set_yticks(range(len(top_agents)))
    ax4.set_yticklabels(top_agents)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax4)
    cbar.set_label('Test Rate', rotation=270, labelpad=15)

plt.tight_layout()
plt.savefig(figures_dir / 'test_contribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"🧪 Test Contribution Analysis Complete!")
print(f"📁 Saved: {figures_dir / 'test_contribution_analysis.png'}")
print(f"⚠️  Overall Test Rate: {overall_test_rate:.1f}% - Lower than industry standards!")

## 3. Interactive Executive Dashboard
### Dynamic visualizations for comprehensive MSR analysis exploration

In [ ]:
# Interactive Executive Dashboard with Plotly
from plotly.subplots import make_subplots

# Create comprehensive dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Agent Market Share', 'Test Contribution Rates', 
                   'PR Volume Trends', 'Test vs Non-Test Distribution',
                   'Agent Performance Metrics', 'Quality Score Matrix'),
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "heatmap"}]]
)

# 1. Agent Market Share (Pie Chart)
agent_counts = df['agent'].value_counts()
fig.add_trace(
    go.Pie(labels=agent_counts.index, values=agent_counts.values,
           name="Market Share", textinfo='label+percent',
           marker_colors=px.colors.qualitative.Set3),
    row=1, col=1
)

# 2. Test Contribution Rates (Bar Chart)
test_rates = df.groupby('agent')['is_test_pr'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=test_rates.index, y=test_rates.values*100,
           name="Test Rate (%)", 
           marker_color=px.colors.sequential.RdYlGn,
           text=[f'{rate:.1f}%' for rate in test_rates.values*100],
           textposition='outside'),
    row=1, col=2
)

# 3. PR Volume Trends Over Time
if 'created_at' in df.columns:
    df['date'] = pd.to_datetime(df['created_at'])
    df['month'] = df['date'].dt.to_period('M')
    monthly_trends = df.groupby(['month', 'agent']).size().unstack(fill_value=0)
    
    for i, agent in enumerate(monthly_trends.columns[:5]):
        fig.add_trace(
            go.Scatter(x=[str(m) for m in monthly_trends.index], 
                      y=monthly_trends[agent],
                      mode='lines+markers', name=agent,
                      line=dict(width=3)),
            row=2, col=1
        )

# 4. Test vs Non-Test Distribution by Top Agents
top_agents = df['agent'].value_counts().head(6).index
agent_test_data = df[df['agent'].isin(top_agents)].groupby('agent')['is_test_pr'].agg(['sum', 'count']).reset_index()
agent_test_data['non_test'] = agent_test_data['count'] - agent_test_data['sum']

fig.add_trace(
    go.Bar(x=agent_test_data['agent'], y=agent_test_data['sum'],
           name='Test PRs', marker_color='lightgreen'),
    row=2, col=2
)
fig.add_trace(
    go.Bar(x=agent_test_data['agent'], y=agent_test_data['non_test'],
           name='Non-Test PRs', marker_color='lightcoral'),
    row=2, col=2
)

# 5. Agent Performance Metrics
performance_metrics = df.groupby('agent').agg({
    'is_test_pr': 'mean',
    'title': lambda x: x.str.len().mean(),  # Average title length
    'body': lambda x: x.str.len().mean()    # Average body length
}).round(3)

performance_metrics['composite_score'] = (
    performance_metrics['is_test_pr'] * 0.6 +  # Test rate weight
    (performance_metrics['title'] / performance_metrics['title'].max()) * 0.2 +  # Title quality
    (performance_metrics['body'] / performance_metrics['body'].max()) * 0.2   # Body quality
)

top_performers = performance_metrics.sort_values('composite_score', ascending=False).head(8)

fig.add_trace(
    go.Bar(x=top_performers.index, y=top_performers['composite_score'],
           name='Performance Score', 
           marker_color=px.colors.sequential.Viridis,
           text=[f'{score:.3f}' for score in top_performers['composite_score']],
           textposition='outside'),
    row=3, col=1
)

# 6. Quality Score Heatmap
if len(top_performers) >= 3:
    metrics_matrix = top_performers[['is_test_pr', 'title', 'body']].values
    metrics_names = ['Test Rate', 'Avg Title Length', 'Avg Body Length']
    
    fig.add_trace(
        go.Heatmap(z=metrics_matrix.T, 
                  x=top_performers.index,
                  y=metrics_names,
                  colorscale='RdYlGn',
                  showscale=True),
        row=3, col=2
    )

# Update layout
fig.update_layout(
    height=1200,
    title_text="🎯 MSR Project: Interactive Executive Dashboard",
    title_font_size=24,
    showlegend=True,
    template="plotly_white"
)

# Update specific subplot properties
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)
fig.update_xaxes(tickangle=45, row=3, col=1)

fig.show()

# Save interactive dashboard
fig.write_html(figures_dir / 'interactive_executive_dashboard.html')
print(f"🎯 Interactive Executive Dashboard Complete!")
print(f"📁 Saved: {figures_dir / 'interactive_executive_dashboard.html'}")

# Display key insights
print(f"\n📊 Key Insights:")
print(f"  • Total Agents Analyzed: {df['agent'].nunique()}")
print(f"  • Overall Test Rate: {df['is_test_pr'].mean():.1%}")
print(f"  • Most Active Agent: {agent_counts.index[0]} ({agent_counts.iloc[0]:,} PRs)")
print(f"  • Highest Test Rate: {test_rates.index[0]} ({test_rates.iloc[0]:.1%})")
print(f"  • Top Performer: {top_performers.index[0]} (Score: {top_performers.iloc[0]['composite_score']:.3f})")

## 4. Advanced Statistical Visualizations
### Deep dive into test-to-code ratios and quality metrics

In [ ]:
# Advanced Statistical Analysis and Visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 15))
fig.suptitle('📈 Advanced Statistical Analysis - Test-to-Code Ratios & Quality Metrics', 
             fontsize=20, fontweight='bold')

# 1. Test-to-Code Ratio Distribution
ratio_data = calculate_test_code_ratios(df)
agent_ratios = [data['test_ratio'] for data in ratio_data['by_agent'].values()]
agent_names = list(ratio_data['by_agent'].keys())

# Create violin plot for ratio distribution
parts = ax1.violinplot(agent_ratios, positions=range(len(agent_names)), 
                       showmeans=True, showmedians=True)
ax1.set_title('Test Ratio Distribution by Agent', fontsize=14, fontweight='bold')
ax1.set_xlabel('AI Agents')
ax1.set_ylabel('Test Ratio')
ax1.set_xticks(range(len(agent_names)))
ax1.set_xticklabels(agent_names, rotation=45, ha='right')

# Color the violin plots
for pc, color in zip(parts['bodies'], plt.cm.viridis(np.linspace(0, 1, len(agent_names)))):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)

# Add overall average line
overall_ratio = ratio_data['overall']['test_ratio']
ax1.axhline(y=overall_ratio, color='red', linestyle='--', 
           label=f'Overall Average ({overall_ratio:.3f})')
ax1.legend()

# 2. Correlation Matrix Heatmap
numeric_cols = ['is_test_pr']
if 'title' in df.columns:
    df['title_length'] = df['title'].str.len()
    numeric_cols.append('title_length')
if 'body' in df.columns:
    df['body_length'] = df['body'].str.len()
    numeric_cols.append('body_length')

# Create agent encoding for correlation
agent_encoded = pd.get_dummies(df['agent'], prefix='agent')
correlation_data = pd.concat([df[numeric_cols], agent_encoded], axis=1)
corr_matrix = correlation_data.corr()

# Plot correlation heatmap
im = ax2.imshow(corr_matrix.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax2.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
ax2.set_xticks(range(len(corr_matrix.columns)))
ax2.set_yticks(range(len(corr_matrix.columns)))
ax2.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax2.set_yticklabels(corr_matrix.columns)

# Add correlation values to heatmap
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        text = ax2.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=8)

plt.colorbar(im, ax=ax2, label='Correlation Coefficient')

# 3. Test Rate vs PR Volume Scatter Plot
agent_stats = df.groupby('agent').agg({
    'is_test_pr': ['mean', 'count'],
    'title': lambda x: x.str.len().mean() if 'title' in df.columns else 0
}).round(3)

agent_stats.columns = ['test_rate', 'pr_count', 'avg_title_length']
agent_stats = agent_stats.reset_index()

scatter = ax3.scatter(agent_stats['pr_count'], agent_stats['test_rate']*100, 
                     s=agent_stats['avg_title_length']*2, 
                     c=agent_stats['test_rate'], cmap='RdYlGn',
                     alpha=0.7, edgecolors='black', linewidth=1)

ax3.set_title('Test Rate vs PR Volume (Bubble Size = Avg Title Length)', 
              fontsize=14, fontweight='bold')
ax3.set_xlabel('Number of PRs')
ax3.set_ylabel('Test Rate (%)')

# Add agent labels
for i, row in agent_stats.iterrows():
    ax3.annotate(row['agent'], (row['pr_count'], row['test_rate']*100),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.colorbar(scatter, ax=ax3, label='Test Rate')

# 4. Quality Score Distribution
# Calculate composite quality scores
if 'title_length' in df.columns and 'body_length' in df.columns:
    quality_scores = df.groupby('agent').agg({
        'is_test_pr': 'mean',
        'title_length': 'mean',
        'body_length': 'mean'
    }).reset_index()
    
    # Normalize scores to 0-1 range
    for col in ['title_length', 'body_length']:
        quality_scores[f'{col}_norm'] = (quality_scores[col] - quality_scores[col].min()) / \
                                       (quality_scores[col].max() - quality_scores[col].min())
    
    # Composite score calculation
    quality_scores['composite_score'] = (
        quality_scores['is_test_pr'] * 0.5 +
        quality_scores['title_length_norm'] * 0.25 +
        quality_scores['body_length_norm'] * 0.25
    )
    
    # Box plot of quality scores
    quality_data = [quality_scores['is_test_pr'], 
                   quality_scores['title_length_norm'],
                   quality_scores['body_length_norm'],
                   quality_scores['composite_score']]
    
    box_plot = ax4.boxplot(quality_data, labels=['Test Rate', 'Title Quality', 
                                                'Body Quality', 'Composite Score'],
                          patch_artist=True, notch=True)
    
    # Color the boxes
    colors = ['lightcoral', 'lightblue', 'lightgreen', 'gold']
    for patch, color in zip(box_plot['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax4.set_title('Quality Score Distribution', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Normalized Score')
    ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(figures_dir / 'advanced_statistical_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"📈 Advanced Statistical Analysis Complete!")
print(f"📁 Saved: {figures_dir / 'advanced_statistical_analysis.png'}")

# Statistical summary
print(f"\n📊 Statistical Summary:")
print(f"  • Mean Test Ratio: {np.mean(agent_ratios):.3f} ± {np.std(agent_ratios):.3f}")
print(f"  • Median Test Ratio: {np.median(agent_ratios):.3f}")
print(f"  • Range: {np.min(agent_ratios):.3f} - {np.max(agent_ratios):.3f}")
if 'composite_score' in locals():
    print(f"  • Best Quality Score: {quality_scores['composite_score'].max():.3f}")
    print(f"  • Worst Quality Score: {quality_scores['composite_score'].min():.3f}")

## 5. Export and Save High-Quality Visualizations
### Professional publication-ready figures with multiple format support

In [ ]:
# Export High-Quality Visualizations
print("💾 Exporting publication-ready visualizations...")

# Create a comprehensive summary figure
fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(24, 16))
fig.suptitle('MSR Project: AI Agents in Software Development - Complete Analysis Summary', 
             fontsize=24, fontweight='bold', y=0.98)

# 1. Agent Market Share
agent_counts = df['agent'].value_counts().head(8)
colors = plt.cm.Set3(np.linspace(0, 1, len(agent_counts)))
wedges, texts, autotexts = ax1.pie(agent_counts.values, labels=agent_counts.index, 
                                  autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Agent Market Share', fontsize=16, fontweight='bold', pad=20)

# 2. Test Contribution Rates
test_rates = df.groupby('agent')['is_test_pr'].mean().sort_values(ascending=False).head(8)
bars = ax2.bar(range(len(test_rates)), test_rates.values*100, 
               color=plt.cm.RdYlGn(test_rates.values))
ax2.set_title('Test Contribution Rate by Agent', fontsize=16, fontweight='bold', pad=20)
ax2.set_ylabel('Test Rate (%)')
ax2.set_xticks(range(len(test_rates)))
ax2.set_xticklabels(test_rates.index, rotation=45, ha='right')

# Add value labels
for bar, rate in zip(bars, test_rates.values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{rate*100:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. Overall Test vs Non-Test Distribution
test_counts = df['is_test_pr'].value_counts()
labels = ['Non-Test PRs', 'Test PRs']
colors_pie = ['lightcoral', 'lightgreen']
explode = (0.05, 0)
ax3.pie(test_counts.values, labels=labels, autopct='%1.1f%%', 
        colors=colors_pie, explode=explode, startangle=90, shadow=True)
ax3.set_title('Overall Test Distribution', fontsize=16, fontweight='bold', pad=20)

# 4. Monthly Trends (if date available)
if 'created_at' in df.columns:
    df['date'] = pd.to_datetime(df['created_at'])
    df['month'] = df['date'].dt.to_period('M')
    monthly_trends = df.groupby(['month', 'agent']).size().unstack(fill_value=0)
    
    # Plot top 5 agents
    top_5_agents = df['agent'].value_counts().head(5).index
    for agent in top_5_agents:
        if agent in monthly_trends.columns:
            ax4.plot(monthly_trends.index.astype(str), monthly_trends[agent], 
                    marker='o', linewidth=2, label=agent, markersize=4)
    
    ax4.set_title('Agent Adoption Trends', fontsize=16, fontweight='bold', pad=20)
    ax4.set_xlabel('Month')
    ax4.set_ylabel('Number of PRs')
    ax4.legend(loc='upper left', fontsize=10)
    ax4.tick_params(axis='x', rotation=45)

# 5. Quality Metrics Comparison
agent_stats = df.groupby('agent').agg({
    'is_test_pr': 'mean',
    'title': lambda x: x.str.len().mean() if len(x) > 0 else 0,
    'body': lambda x: x.str.len().mean() if len(x) > 0 else 0
}).head(8)

x_pos = np.arange(len(agent_stats))
width = 0.25

bars1 = ax5.bar(x_pos - width, agent_stats['is_test_pr']*100, width, 
                label='Test Rate (%)', color='lightgreen', alpha=0.8)
bars2 = ax5.bar(x_pos, agent_stats['title']/10, width, 
                label='Avg Title Length/10', color='lightblue', alpha=0.8)
bars3 = ax5.bar(x_pos + width, agent_stats['body']/100, width, 
                label='Avg Body Length/100', color='lightcoral', alpha=0.8)

ax5.set_title('Quality Metrics by Agent', fontsize=16, fontweight='bold', pad=20)
ax5.set_xlabel('AI Agents')
ax5.set_ylabel('Normalized Metrics')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(agent_stats.index, rotation=45, ha='right')
ax5.legend()

# 6. Key Performance Indicators
kpi_data = {
    'Total PRs': len(df),
    'Unique Agents': df['agent'].nunique(),
    'Overall Test Rate': df['is_test_pr'].mean(),
    'Top Agent Share': df['agent'].value_counts().iloc[0] / len(df),
    'Avg Title Length': df['title'].str.len().mean() if 'title' in df.columns else 0,
    'Avg Body Length': df['body'].str.len().mean() if 'body' in df.columns else 0
}

# Create KPI text display
ax6.axis('off')
kpi_text = []
for i, (key, value) in enumerate(kpi_data.items()):
    if 'Rate' in key or 'Share' in key:
        kpi_text.append(f"{key}: {value:.1%}")
    elif 'Length' in key:
        kpi_text.append(f"{key}: {value:.0f} chars")
    else:
        kpi_text.append(f"{key}: {value:,}")

# Display KPIs in a formatted way
kpi_string = "\n".join(kpi_text)
ax6.text(0.5, 0.7, "📊 Key Performance Indicators", 
         fontsize=18, fontweight='bold', ha='center', va='center', 
         transform=ax6.transAxes)
ax6.text(0.5, 0.4, kpi_string, 
         fontsize=14, ha='center', va='center', 
         transform=ax6.transAxes, bbox=dict(boxstyle="round,pad=0.3", 
         facecolor="lightgray", alpha=0.8))

# Add warning box
warning_text = "⚠️ CAUTION: Low overall test rate suggests\ninsufficient quality assurance practices"
ax6.text(0.5, 0.1, warning_text, 
         fontsize=12, ha='center', va='center', 
         transform=ax6.transAxes, bbox=dict(boxstyle="round,pad=0.3", 
         facecolor="yellow", alpha=0.8))

plt.tight_layout()

# Save in multiple formats for different use cases
formats = {
    'png': {'dpi': 300, 'bbox_inches': 'tight'},
    'pdf': {'bbox_inches': 'tight'},
    'svg': {'bbox_inches': 'tight'},
    'jpg': {'dpi': 300, 'bbox_inches': 'tight', 'facecolor': 'white'}
}

for fmt, kwargs in formats.items():
    filename = figures_dir / f'msr_complete_analysis.{fmt}'
    plt.savefig(filename, **kwargs)
    print(f"💾 Saved: {filename}")

plt.show()

# Create a summary report
summary_report = {
    'analysis_timestamp': datetime.now().isoformat(),
    'dataset_summary': {
        'total_prs': len(df),
        'unique_agents': df['agent'].nunique(),
        'date_range': {
            'start': str(df['created_at'].min()) if 'created_at' in df.columns else 'N/A',
            'end': str(df['created_at'].max()) if 'created_at' in df.columns else 'N/A'
        }
    },
    'key_findings': {
        'overall_test_rate': float(df['is_test_pr'].mean()),
        'top_agent': {
            'name': agent_counts.index[0],
            'prs': int(agent_counts.iloc[0]),
            'market_share': float(agent_counts.iloc[0] / len(df))
        },
        'best_test_rate_agent': {
            'name': test_rates.index[0],
            'test_rate': float(test_rates.iloc[0])
        }
    },
    'visualizations_generated': [
        'agent_distribution_analysis.png',
        'test_contribution_analysis.png',
        'interactive_executive_dashboard.html',
        'advanced_statistical_analysis.png',
        'msr_complete_analysis.png',
        'msr_complete_analysis.pdf',
        'msr_complete_analysis.svg',
        'msr_complete_analysis.jpg'
    ]
}

# Save summary report
with open(figures_dir / 'visualization_summary_report.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\n🎯 Visualization Recreation Complete!")
print(f"📁 All files saved to: {figures_dir}")
print(f"📊 Generated {len(summary_report['visualizations_generated'])} visualization files")
print(f"📈 Analysis covers {len(df):,} PRs from {df['agent'].nunique()} AI agents")
print(f"⚠️  Key Finding: {df['is_test_pr'].mean():.1%} test rate - needs improvement!")